# FinDisputeEval CFPB Seed Pool Build v05.1

Builds a **candidate**, not a benchmark release. It reads the frozen EDA v05.1 run and hash-bound decision record v03, writes directly to Google Drive, and leaves EDA and Seed v05 unchanged. Formal generation remains blocked until full NER and manual privacy QA pass.


In [ ]:
from pathlib import Path
import hashlib
import json
import sys

IN_COLAB = "google.colab" in sys.modules
RUN_ID = "run_20260713T145423Z"
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/FinDisputeEval")
else:
    here = Path.cwd().resolve()
    ROOT = next((p for p in (here, *here.parents) if (p / "WORK_PROGRESS.md").exists()), None)
    if ROOT is None:
        raise FileNotFoundError("Open the FinDisputeEval repository in VS Code.")

EDA_RUN_DIR = ROOT / "outputs/data_pipeline/cfpb_seed_source_eda/eda_v051" / RUN_ID
AUDIT_ROOT = ROOT / "dataset/curated/annotations/cfpb_seed_v05_audit" / RUN_ID
DECISION_RECORD = AUDIT_ROOT / "seed_v051_decision_record_v03.json"
OUTPUT_DIR = ROOT / "dataset/curated/seed_pools/cfpb_dispute/seed_v051"
print(f"Runtime: {'Colab' if IN_COLAB else 'local VS Code'}")
print(f"Frozen EDA: {EDA_RUN_DIR}")
print(f"Governance: {DECISION_RECORD}")
print(f"Direct persistent output: {OUTPUT_DIR}")


## Required Drive migration

Before running, the EDA run must exist under `MyDrive/FinDisputeEval/outputs/...`, and the audit directory must contain decision v03, row overrides, PII clearance, and fuzzy adjudication. This cell fails closed on any missing or changed file.


In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

required = [
    EDA_RUN_DIR / "manifest.json",
    EDA_RUN_DIR / "analysis_ready_corpus.parquet",
    DECISION_RECORD,
    AUDIT_ROOT / "seed_v051_row_overrides_v01.csv",
    AUDIT_ROOT / "pii_release_clearance_v01.csv",
    AUDIT_ROOT / "fuzzy_duplicate_audit_master.csv",
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Missing persistent inputs:\n" + "\n".join(missing))

decision = json.loads(DECISION_RECORD.read_text(encoding="utf-8"))
if decision.get("record_version") != "v03" or not decision.get("release_gate", {}).get("passed"):
    raise ValueError("Decision record v03 candidate-build gate is not passed")
eda_hash = sha256_file(EDA_RUN_DIR / "manifest.json")
if eda_hash != decision["source_manifest_sha256"]:
    raise ValueError("Frozen EDA manifest hash mismatch")
for filename, metadata in decision["governance_inputs"].items():
    path = AUDIT_ROOT / filename
    if not path.exists() or sha256_file(path) != metadata["sha256"]:
        raise ValueError(f"Governance hash mismatch: {filename}")
print("Pinned-input preflight: PASS")
print("Benchmark release gate:", decision["benchmark_release_gate"]["passed"])


In [ ]:
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas>=2.2,<3", "numpy>=1.26,<3", "pyarrow>=16"])


In [ ]:
import base64
import gzip

RUNTIME_SRC = Path("/content/FinDisputeEval_runtime_src") if IN_COLAB else ROOT / ".runtime/seed_v051"
PACKAGE = RUNTIME_SRC / "findisputeeval/curation"
PACKAGE.mkdir(parents=True, exist_ok=True)
(RUNTIME_SRC / "findisputeeval/__init__.py").write_text("", encoding="utf-8")
(PACKAGE / "__init__.py").write_text("", encoding="utf-8")

embedded = {
    "cfpb_seed_v05.py": ("1f0fcfac9dda5eb8b122fc3007b43ad3f185c3667b9c309cca89cfeef7c0d23a", "H4sIAAAAAAAC/61cfXPbxtH/n58CQScdoKZQyo3TVmPWdW15qk5qu37JPE8pDAYkjyRiEIDxIpmRlc/e3b134PiiTDwZhTzc7e3u7e3+du9A3/dfskXWZGVxtk5btvRevHr7D+89g083kydek26rnNXRaPRhw+Q3LytaVrQwJs3znbdJG68ovS1rN+WyzMt1tkhzb8lWaZe3TeRdtd6iLJpuyxqvLGBAOqrSphEzpN0ya6E3Z8Kr2aKsl97tpmyYly4WrEKm6vK2QSJtmhXeNl1ssoKd1SxdpvOcjf71/s1rr+5yxicr2A3wiE9hPmR2w+qsbbwWJLiZfOc1KFzNcpY2LBr5vj8arepy6yXJqmu7miWJl22rsm69tCjKNkVBm9FItIG0mzyby68/NWUhP9eME1qmbbrIUcRGUlJNqgdrsy0zHtP3sYd/fy4LQalKW5xMdnsLX/mDdldlxVq2Py92ir+i21agYliSSjZVabGEBvivWo5Go3eX//l49e7yZfLy8sXV+6s3r5Pn7y6fv/em3t3Ig38++5Iu2mTZVTmsZMsaf8zbWwbLDw3JKt1meaYfrLqff945BtRsnTUtq+V3WJdd0myApyQvi7XutoQJQcuyocoy+TFPi3WXrpn6DmJ3QDRbJCuW4nKp2cg64XHyuYNVw+Z7FPYHkO0y+XD5fx+SF29++Pjv14akRVrXsL43LOmK2zqtKraU1PQjMEjmaE2LskBLHz7K2Rfx4H70/s3Hdy/2Tf+Cb4saTBs1C9vKU1SGZOv0dtj4W/K9TdvF5pA49OC/V2+B3yUj8d5eXSVvn3/4cPnOlOvy38+vfvAvYENEKFmWs6D2r+ez52f/nZz9NUq+fXQWP/q7/AqfryP8Et89Ht9fz/0xDrwKxXxv//nm9aVNjB7gv9oPnj395noZBs8urh89O5+dRddN/Cx8ht+DZ9fLuz/dX4fPZDN9F1/g83f3wTMc7BM9Od/796/7rItJaPgZ/H1Mf9VwOfL5ixdvPr7+sJfX6zmwBT6t7Ir2K/y//bpIayDb/AHaYd/OWf21KK+jZ19/h40wxfdcH5oG6GVscXv19sfvhppG8Zd352OQPgpRB+LLHHmFXTH6u3ZI9Jc8/o+TJ2CRq2x9QbRhVyZ1VyTLrL7grgdbpadOuKdO0EUZj8uurbq2N6YGB1RuE3S8Fxg8wFIeTx5/P/nz+Z9GgugK3G9VNm2SFVmbJEHD8lXonf3New3O8EIpAJsjgzGghLME/fYwAsdQ5jcsCO2xLvYtIq4Oe6lpaS0autkcORqhmM0mffzk+2SFS6V1N/YWm674lDTZz0yq6Nx7+hT0RFpo2porYZmtWYNPRRyKOD3B2G0G0hAfnO2yYkXg17Ds6P2BCEu3Wpm3G2CCT+xdTMXjCANnoLkJdX89f9RVGLJ4Nz51zcAZF/L5hn3hn5TcsPo5Iy+ffGK7QNnCWDu/JIMm4MKWuEp3eZkuQeSVf4fD7r/emUPu/YgV6JACv2tXZ3/xLX56ahLEQgeD2J5IxJEQnDAWiHhqIcKx2TJbtDNgb4xxNx579vf4QkxPOGZKACFC2k1gLAwqOWnZlzYg3iFqTSX3nP1sBZiqFWSiNWthGTlkSRCngY+8uw95OwdT0PIqzRtzveo0AxD1Y5p37LKuyzrwFa4TtDykJQAc4gQipBgwJ2/Krl6wBLcYRHHcIGBT30whpk+eROf+iZN+7jLYD17q0ag+6hMzy9YGtGeyoNpB1FnM+853SQq6xNADGHGm+lCrH18gcvRWgALx/4AdFY17Gr7NmgZxFBg/gBK2DPZhozPY720gZtNLJMYfkH6l8LU0iaxR0yKx5sK7E9/vpemi7V307ApF5EyjOCQ0yHOYbdMWQP6pVNcM/8TqGQgC0wTQxVZ00gD47VDdPpglGPEtqwO+6nKf+LZ3OCR+xo1MjgSxkQspM/67wXHe1GZGoH3akAluJskO9MoqwyGLHUNETudKJi/9WRzstfWuRxfXievS3ujEgx7IvqDIvAPmKjA5uCtiBd0yww9HOb4qgGq2HHJK9iCZ9Sg7IIqmF+SmN+YcS4dM9BDOEynMrmqA4YHT+nqhWOzkJbryRuE+d/IAmoRF/JJUrE74Q/TJ/r1GNY7EwhxEjTtrwCDhwP4Ag/MdyrHK6i3DyJ3VjTVM5SPUPUeDBk8vGsnOS0gS60RkI+bQfuqCFIw2mb+IjtjFRUVnOkrAsupySjGTqi4xXRN00nkJ2LvdgL/clPnSRQ3TpAtD91JG9iWps+ZTYiVVRgdy/Tz8gLrybjvow1UpSPWyM9VpCfxC/uGi1xh9TY5VNmcuAPcy7LD+XXkfElFJn2o1R/VTwoG2qjrbpriKDBkBuNMXkhV1tthsodyh00qrA6KmpgHhAbu6CMBzWN7VLpnvhkqxHfrYo+2UFfb+ijLYH01gOPNB2KJxPEYZTim0nOMgVu1xNNyVkKswYxVOMQxVfKXIBYBhYXkGVxEcXt5hhqjUg3Uff5+0d4e22ezEbRbv2WazfTspHuykmWPzxG7rndmWGrudjLvX/XBBUSegDVQG6dlaI0RielFn0CGmSAedhfrt/nvW9SPUGqqK7EWMA8zUo/tNfW+Ek+gO2uRKg4ooUCAoo1GktFgiIfl45vYvimNXPeYQeAS6kYuighOghAo85w1IhY+9m4zd+haGhkBfgI8pFpBwKS7dHg7Q/Lws8/AYR87R3raDzGzOiARLC82Fnvagz4ylUB9qE8IcQNQ0I9L0NE2PQ/Uz0oYkrVjR9QhzxQZWTyBPbmM1BNJ4nigcUyMvURwQwfaR78hzAOtgetxhnFGhmOh6yFgjXaJsU3MpSjLtEw/0hsfMEXCZdI2Gd4hnB2JwbBnRJJp4T6dO8tB8Hk0OrZcaFe2dzjCerliCKmaTsXce25asBBAeBvgfopj4MCe8WzQYxxnAWYAJwGPtzjm38oHG3CqAH5xajoz6w47NbOxfyYQDDAA/QywA+zmHfgf3s4NYNKCkliclgscYHODTeLYHnp7icfrUIiepPe7HCre9aOvA6i6ofgSp94H63igH6oKSTdAPOqH31Ds/EZbwkGSsBtQMM6xS05nUGvYEl5pjNR2r+kAwHvXCbuCCgg50NxSHU1SSTI5K0uOlJxJU588KtuZnAPvMTMzpQKeiHnXQnvoMDKho/RZeOf8Jynaak7TYBSg2z3FRYFIjz91BkQdYi6gTmMRB7i7VQK/HzYNUY+Ju6QYwvmB3Z5+HaMwYaNgi8kf+S/oIVfmUyJXHvVWdbqHKWy2jl1CFf4XfxpDXN5+o7T2cVrJmLDAaFUTxQCRtxJdRLxdPVytYIIZhDml4v+f0QTo0ZBnpIvY5kAg91FxAwWIxkxTQ3s0xQJF/PKl/wnmkYfyjkJ97K5Zn62wOPqPdieLqfl04SxBcbrOjrEU0cM6MQZ4YXJTVLgiNJ7OBUFIP+zqZkvi8V86KdbvpAeB+ssKdymC59Szj3veZ/7lLSSNwKrfkxRhNFBYNTgnyIg14YVePNrhxpUexs24h/HL4mzHJZT6dx33ZmiZmcqiRlq30Psp8kCSaKESxBUP4BnHLBH3RurXBqYCOFjcnwEbjjG4ozUk5qR7S6/zrF5JwuwL7+9fOTPCOJcU9miZvEuhJ5IfxWJcojkDJB4j3i5QP6pmyZh1HGBOCAQtDKU029ib1spMgn0AUSQRtp8gaVlsiH0Huv0ZmOVzvzr7saoKh7JofXmy35O9Vb0JHjWOvHlTdGZkUcWCRVhprJuu67KoHB4Hx4Cz5xLAAR1vZ2tCnfRw5FSHzhbqHcfUSdLJNK9sZ5Ol2vkzFGUP/SNPgbExnGBwihb0MFU4u0FcLLOTbRERs5uAbFAb8a8g8xOsygbXxurAjDseHVIYY3iAjMLxOfjUrgNIRQBlkLdw+RE0vJaO49BrG9SF744f2ivEPBDV69h+R1cx3ga+usfASCj/e9cNo0W3pfkUA51VwuCqZ5zPED59HSIvGQ5ngYAqtDnsOt/3rnI0ndVBgoa25fx+oE8HBVjhyHWO8b2cIzIxpYmBWcQapqQnTuTzEoDiVz+rEOrADStyJpMuf4E4frrugREy5ju0EJ5rWyRWjfkpsnC07Z4e7DqzwlIQez5ghpSR0moH36u1S4sm8HKKZHPKO/cDnQqCGux4d3IswNGcSci1WVAEX4HD+qHu66cMtrKa18mmuo1dwieN12b7CuhFXlRqjRcE4ZJT+lvwCAkz9uYMl6w2QVUs1ddOtVtkXdQA8BVgcibFabwwwhKK8aG40VUhJ4b4im2I4yRDmQdLNIBbwG6JJkU45ALHiBz8AoYMF87ZHkqIhWS14YQz8VwEoNYFEDZCScHD+nkN+RZyfmJB+IlEpfdjZ/nNtbB6Z5BkS8/Byj3l2ImgPj0+0QU75MtnOiJpmbtliCGmo1QAvzOCuksvDg7+1XcDnQ0UZ1bTjx2znftyz91iYCtqi6Wngj7h6oC5qrbJiGYgQOLito6lEoFuxxLz32Oudi5ONYdcZPaCis+P43u4z5fNbjWHfT1lPNeddgZgqZ6tWptXZetNqKey7ZtgvqcuyFf3os5wfH4Zj/pmeWmKpoSiSHuyUCyJv4JwqRGG3WbHn6UiVsPC5eIQVGGVU2pBmx/cQN4AYq3Y13XNqoMyzZF/43hzDxdQtm6KCDAfE9YneX6iDAgFpIzwxN98TEEUVgfc5hMpsREbLYUAvebVFFKsyaRncWUEpDvZizlqorNxRl3vf8EE2cJK3UThM0mxLhqFar33HCeBoL6UBTvIdkPoIONqj1D52MUCLKFXxpFMkdJDaBYiuLkQR0N7pam1RwdhN15vx5YB2N8br6zBtgXo3LwoPK8OKlhgBEWcO7nUGV5qev/gA95ruOMn7mC4HY+f9eUZVM1zlRN6XI9s7ArDkiZYbZGUFHYQlXBkInIBVPKQbP7woJS7SiVPM6dGD05EqNRs8WFe3LILimFVYhAw8x0rS78R1QOPM0IhfELesScxLUVb5DIebO9capaoNAAPN2GWFg2MHiz1JDk1utpLPGBi3a1/hYar2LkJ/05nQCRq3VE/hue724+ZXHThNo45AF7OaqQ/ZhL6cHyq7lUUkDPa8wpxhTYTeujliwAW/TMtNDMemF1SFRlMWDPQz54MJAp5qTijdIwOmsrY7HYCrtHBheBKaRk4U/jbFImBAnfakEuaYAUxDbMZFCQVSk2N+BVJ7Lyr4QjkE/eWMdGcOAHnVOSDabVl/4mwNOBaPZj5frW5L5idb+UQKpknwG1rbYL0O/K9+9FOZFVDxB4Q/PZfIELx0Y5AzJuHnKAnvEuBlYgzOkHjdpFmOb0QB5bYUfs8kN/OzJYM3OZBN2aIHxZCD4IKJGUPvD15hDcZCD2FdIlBUEdRGQbk92mpzg4mFB8aLVrMZr71pdmDXYhXLUErNtimCErxtX4BNYGx2EIfosQ2EeUCaRd37XMJox0i7NpM2C1bQ3Wyen0QEiFSUE+uBG51msbyyZhU3ku225pAifTI7c06itJ0JmqgIQ1lP3T20qmz6x8g9ggsLPT8qmT2TjyCCkvnN9ElpuVrBJhx7gaIKl3QJVLEC32KiFwL41Agi8Z0966ivTT8xCajwrrapd1Mb2M8WiICN3gsEflz7As64xJcw7kPtlo7NYT0DohYJr1pMcTpdVoTyLpsantJ7JOQOrWgBThNAFnAfEOmxxz16wiEzQoOQhxIZQTSb0t3PuwwK/xSk4CpPsOBv3tgv4pCD1ogECxFirXulBFEuBR25XmPgtN2vtvCS86G7wKF6GwgS0WyFR+GiniHomu/j/JFSceoV0W3twWD7qvSA7EmvRpijODD/7V5KAMjpSdqqHFvRgWeiNCjSiEFJx3pZQsklq5O6prqHHvBuvh80UM7Bg/P+iwbLkvHrZPReH73+ymvNILlDSB4AEZH2qkPuVU4BpewaKHphXywe4mhVFVKiYkDhz2hlHk+eJE/+8teDofpSaAc7j6Gz5Awdythbg0B3BtWL8b0lAJRLwHo3FGJAEoiD8uVaE9bxntzi2YJBlkXy4usvcMYgYdqiZLCOEJa6dkGbWhTwly0WXFZE1P/2/8++3fZ1uO/Mm3cQu1V5FH75bSqGk3uTLO4/xI/tdFoTcR2zyOdi5rHcuoancxI6XK82BwzOloZ3oMbGAux1SOYp0mm3eASimxLedd//EIVQdWBLWQLnm7Qtv8CZpz7U5atHSIQP5xeE0K5OQOr2hFp048qQ48pRbJyycbkGSjOPmQxdGfdtDsZudbNNnLKB+wMfAMtvB3LNBQfjGIQOXTsKZVIf9u9+qnPJpq91K0T/cmQNrM6/93RvWyKBAqw2jS40lb66JDiwYcOJ6zwUdTx4RrBHKXv43J1iCtu2sko9i0yF4iG5obEAkjmfJJPJREEae1T/nFJrqH92IJBPX4UuEGTiun5/++RAWYYjnRSVE8br5IqBmdg/Y4N0vJcLHC78acYzEN5yenXRX6yqOYK1xAcd9t4s1WXG/uuo1sums4vz72NTKH7Cbm2Sk8KAdTQfG7Qe4J8Gk7vd1OASpOmlTrfbwWwO83Wa7WMy27HDFJyVPn4Ab95hGVvVM9MmtMaclOjxibSkM5ev1nL7olXkhuYOLaadGzvEQeWXk8nwe/v7NdQDQkcsTb4GEMbuG0Km0Ria4ecGZj2dr61+OT7afoK/4iinIVXCVsZjzqT8ZGoWYEHvLUPiUoLNiyFpRKi0W2V+pZDpuEcEc5H8NBK8q3n3ybgBpngJRg4/fICscY9seKhqXrqx/eevnU1TOTyb2K0P0XFiOAqXvqVdPoyoHNWjeK8cAWJ8fZINpjKzzYO2rDrJCq1xuKTWIL7GMKSEW7hFO/U5NMVTUwAUTFopeDtI8KBAlGUm2Z4PcHDmMBkXf0NP4KDlMAinrLQqLi3ZS+zkQyrfMXt/NV3jjaT/btR/BRXfHbB+bso0lgXklZgiQ+oF/WQOFxXlbSB/IymCZ3jcXcJqQI4bWDfUdATRZjaAznwXk/9JrGRbZOwXR9LxIZV+SuMkdKgeY25BXkbAvBcIGDmvOa20ON1JNO1xHUZH3erY+EY/3mJpF90//uwMJZO6ne9jes/WesNV7G/6Ga9DfpxmYxY+tr20q4PTCxuKkDvoqC+1deIaNnCKbvU4HB3RFs2hqRppUDOlvNhSUUEHPv3XlvFCFd1bCHpGw5eHr52h9RmSifuvJrts00UPQR/k0fwNIX4jqMX9FglAaLzXbL3SWlCKScVCOnoHvyHTRIcGelVFOyaocOCoLtr1w1v4sTcOQXRspILjEn4ZrQlkZ0waGvy1N8OJcwcGXv9xiI/t6uPIvBxGHlAV0OiihsmEVSrGzqP/Af1WfghpTwAA"),
    "cfpb_seed_v051.py": ("30cc21291827340b57b55e4a9dd5dd3c7e6d553e5b4c247356b643dc0e1b5a61", "H4sIAAAAAAAC/9U9a4/jRnLf9StoBndHIhrd7Bp2LkpkxNnHYYGcvdnHIYEgEJTUmqFXIrV87O54PPfbU49+s0lJu7aBGPCsRFZXV1dXV1dVV5fiOH6eF/urzb5qxDZ68vzlf0avBXz6cP3N7FG0ycttsc1bEa27Yr8V9WwyeXMr5Fv5LDrWohH1B9FELbzb1dXPooyePf1egh3zWpRtBKiiUnwA+Ar+fKyLVjQT3Vf0oo02Vdl0B0CTl1G+2YhjCy+3YlM0RVVGtdhUNQJ/PY1u8+b2al11gDLvtkUb1dXHCaKti61optRXDvgOx70A4jd7kdd5uYFPArpool1VR0jKHWC9EZ+u6qJ5hzhmkziOJxMYwiHKsl3XdrXIsqg4HKsaR1BWbd4CMc1kIp8hJftirb7+1FQlNweu5Zt93jQwHvlSP9IQoi0OwnpN36cR/v25KgXDHfMWu1BgL+Erv2jvjkV5o55/X95pqo7AgBz42ETHrRzObLM7rrMG+J0Bv1WjZBLBfy9fvMhefv/mzbNXP7ye0pPXP7599eRZ9ubZ/7zJnvz4X2//pl5ktdjmmzYjvmXHouDH+fG4v8vEvrgp1sW+aO+yutvDTNDLTX7MYG53RX2A3nfdzz/fZZt917SilhDHukJygLP5PmvaGpi8KwC2yXEGGaa5zR9/8222K/BBOplMXj3777cvXj17mj199uTF6xc//pB9/+rZ96+jRXRPDWLxCSnddsd9sQHeNjEjilsBWOFBtssPQKx5waT1G8BgC6RWfUfJyZpboDjbV+WNAUPWwCDUA2CP+rjPy5suvxH6O8xcB0iLTbYTOcqZ7o0GDa+z9x2ImyGi+phpETddgmg3Ims2t+KQw9MHYMx/GEGjv7Si/379zaMnOAk3c2oKtMIcldm2qOcsU/hUrbaMV1uGsme9rrr22LVeG1hZ2+pAojWPCljpi+jx9eNvr//l0dcTiXQHq+lYNW1WlEWbZUkj9rs0uvou+gGknOmhKYbHM4swwIS9JP7zdAbsqvYfRJK6bUPkO0hCAIPYzGgdHOax3XIyoWHWIt9mm+ZDYjhHAz1uZ09hVp7X+UEOuBYw6yW+0G1099h4qr9tYaGLRQzrotDChv+9E+KYQa95t2+zMl88z/eNMK9Fuam20GIRd+3u6i9XTaEaW9QS+dkNClaJGhJmCAaXuMJgxjKVpCPz5tG22LRLIGuKymfF73B9ljjICF9MaPDYkgd9EG2O0gkMZSSzG9Emsd8/yHd0/5DSS4WQJ6fYRaCENR4jO3VeNCL6e77vxLO6rupkFz/1to5tBboYW/P2UcAmYTqOqON5dK/6e4i5RylDDjNmck/7sx6uTRxBiE+wupsk9Sl8Dg1+qNrnuHkxoSSDrEU3bZfvoTNL11mvAb2E+GqhOcAM5AZxOsIP/YZmKf6rN3JiSnQomkPebm5tPvxbFHttxaej2MDmvLh3qPgTU/Gn9GEqCV3c878PBkPqCD/qEJbFfQWLQO36sEV9PSKDJFQtqGmx9ETQEkn3zWo1t0QXWIw79Qw7bRJa2k5HKa/JVnxqE28ZxameDFuCpT4BniKSOMUpimEU8ciMxJalVYv3XQHLMWTvSEG8QTts4fXK6h9f8ZJRtOETKRqb6igkQWSxaSPgUabtuzgCiwhl17Q7ou2yHRUpewCEmmksGrkMGEGIXU3V1bDYUaf7LANkFzPNGJtmQcuOFT8bj3X6OfBtuWLY9V0G6xqV0z0IToIGoQNLb0EA5mgrkhGJ/xal6eRBDbaBdhIdDWzIWAEcsMy09DUEjE9U4zFWaAUHQhIhdIN/cRE3aBgCbm3LbLU2qxvhqG5vneDgeRQ4PmIHDLABW0dsk4FR2EQCQxaKkUv8s9LvkC1BtjZgVXc4ETEwFxblR1EnLA5KIVgScUrZS+lTLUGTIRVKmxseMHGuJnDIO+Sb26IUZMlmCCQJTO0BObi+8lYnmsDWTubN58ldC1iJjgy6RdWeluk2IpwwZ01+UwvhDe5DzmC8uAk0YQIdrcs0TiXtUv+G2tLfEVnxTDi2V3GdYbtlz5BlSYDmomkyDcwfpFqw3ymJrYsKnMW7MKh6GzvmATCwBKECLZA4KHkEqVJ2zruxhfYKllZVXhnPRDZVQ8ZVJ5WRpfD2YAG4/fPqxsd6VEAKqJ9Rhcdd6RYH8Jwi8m328AV3a0ACfrhHFZiHd40hBjzrBIzz5ANiTmlx00da3TaNM3oMlgsRi000bQywVLwHD6nNmuJnIWf2nFFIhtEYgKSoraIetlT6e6Bzdvl+7028KOtic3sA8yvTIOBcgJq7c7dA/VYu52qrtkF4Bs5UscvEp9sc/VCwPHbg3zkOLCw92B+Pzehm9BbCCccjKcfIEGbRLgkbImoHwZcMNGBXgk21z0HdbM2ogFqpzN7UnRgnQzcPkiEZjr1FHOpRi6sozHpFd3WlCIUvysYguc50ECUD2ZH+P8ZNzqfyWSDiwoTJPkygxjBM02EiCPAok442z6f4BOEEmN2xzp+o5hjt4F7XIpItt4H+upKoQdm4tK+3qulYX9qXNxPguvh6KvQjZ0K25zP+FfDZdKfo8RQWhxAsYtzQgqaGvztKuNhmlxMlFQKo6fVeRC+eDmlRu79DXhY71BNopDdyRoBQ0MrgxbZVpgBGp+dvEojcD8MPXMZIS6zdY6ZNxQqKLWhtsSs+sWvL4UVYcxDVACujWv8kcGuBHRHecvfb4ga7Wago4YwdJDIz7NbpjJwMkWgXY3YrPnHrJF3OH327crwmoiL6Z4lfUwsTRpKageZPdDhmGqEX1xzzjTib8GN+hwYRUL6L7xHTwy/3Ggt8tjE8xD71Nq3eyCViZ3ySfg4h4trWWoB3HmD+sQMzxA6hyMCiAvRfDoZcGgiVwKgY5Qw8o7sktV4tQU3IkYFExivyGUIvZnmDQZlEBWVSlx7sQn22e9EPAx0Nvgv3BStjuIUx+5N0lpd3ybgboaAjG02DlgHqL18po+qGeUe9hV6OQUycAkt6s1RMw9m0dorVDIIK+zJPKFKVgjkdHikMNJ2YYIHZeUyvJ7mFOIzrqUj+auFiOzdeEjuMcE0weabAMsXbG3XY9AMn0jNb3Et/StF15RCF8ZN+yAVsT93OAb9SWNJemIVpynAR8j7DUwQMhDjbVnxKXOalyxgiAlUJwrCn4Ecm40r9mbGcXyPpoIDRvk7ATgXzYyGDIo0dQ7A1B04lO1z+JDouljOIWd4ubRzkdGksA9Sfdrrc6eXoRC8i5mq91FUpkrUHUd8IIztGTB0K3G8kYA4Dpv33ZsqVt9yHUVu2cwrhQNBpGe6SEJQjXUtnd2V7FixKgN+v8e5XEDPDPcJdOJtq3x3KZnF/xojmdGKSfcZIey2DHHgwq8O8qMrFEO9vq4+LeC92NneUm7yIwe1FiwP+0VF1eyMJjsTeUMIASkPqDmOwpWwderIrOfThrhTAUFfSAJv0orWMztmuHYv11HatAb98u7aNZ2MehwACO+3gu/BOK5W22nxYA+MeNIgHxkObVNh2MNuShXn0IONH2ZG3P6O9nK9x+UZ0xKs4f2/hVXrqd9kJzEz8/90JbF/pko3gpq66o9g68kjP1nceb6ckQwv0j9TclDswwFqcl2WQP/ohMtt+MeV+kfmSAJtf9GgZ6wVfdmXxvhMcW300MfqbeCuJGJXEJxIKw8u17VUSVyQGd29cxgooQ50BwTNahGQChsFqirYRVBxPzhu2oZpHKyXK50CB1um1JVWHHOIQg1a+eN/31dKQwYtowGijkxVLAfNHqwEODKc5/iWe/VQVZWKpE0krA2XoUXnW7Lld2wzcOUGF+b0kwbIU+6iC84Xiek4bq2/Z1+AWksnwAaI4uXmcikXLXcQ8BRSvBaifRp78UdAPeKQyMZQnIcMdOkFjtYwVKG/KGIpRAi0TPLL2Fii/rfbkIoNHq9GYHBDAc8ghRaY6dntKGspMlotBSFwxxpHvMc0j35xC/pxyrJwmf3QaBcwAEHIZEeUgqMSrB5KGzCYz0CxfVxh8URwBmmWHE89kUw1gLW0EgkJnFk8sS/IG984er+3u1XTJOcrQdJHTBgT8Qw0ZjDVjWuGpQNITAxsrJfZQSo8eRPwezF+MB+uz0sxA+Q6th4nir6dRMdgwrq7sGrTBQQvCDjQoE33sB0i5ASMNItSi3oC5PConv2hE6ztoC6hEjYY/HhJlGDlGRXB5+6IkkzkTDQTTz6WgFkeB0YvsCFFYSCBwDVavqWTVg9QqHAaTeshJJiNt8+V6JpTz5OQkOEjpuGcZTkY478CM1CMAueqSRzDlpjKCSyNHruU17OPcs00MKuXlyoVF2xP1IFK3kirRQ8f21ikolUw3AAfnsZDygobDPMgSQ9oaJNUY/bjH6OXs7K+kuFQ4Pl3Z1j/YM6GzQqOKq3J/Zxujsk/8h3rED6AwTQM+GYLg8MrpiMQr9+w29Hg9iw0fobVinaChWN5l67t45Zq4CpRoke6zNNW0LVTtdsDZqdxhEVqUkDkLOEXiHdLpA1LnkBGcPIHnXhGf8HltlPgtGX/fXCIRVG/B72cXgiimj+nAUtWqqbGZjdgcrpK8fACVQ+H3tYAhC0h0g6hmxMkRBs8AbvOFenDUzD/My7HgIW8X9jrx91UH0RnukoeRl9UoUlpTsEEUNyXliI7iUwswtTZSj6k2bqVO4LCA48/DXDnkR3fL2eeH9TbnA+S5f/Jg6UYloVMGTb3wZG/avI3NTCKayZk8mHYod9k324I5YSXwJgPTcqpVj++BBokKbQ3QExJlNrBOSTLm+QWb/LtZuCf82F7EbOccwEddqXsghSL9goe5F3jmprrXxb3+iDl+CgWk+YXofXBxOdm1JLLAg/HM7yF5cON6mib3MetmD9TIJpyfPb7Orq+v4YPUp4G5wzQOYRZZyjk+501Cj+dg8KJepKQkyfC4zxWthnuuVAjU8zhVW9rgxuFdN9FW/XNNXQ8D2xYz8AxFuTVs6cOBvpx1R5ytxPR+6qgmYHb0kZyhaQPYlHbs4xtXstYy1haMGr4z/fehKDUxeK60YAhCihIBBcWYY/H+bripKy0hvNoAQ3jPHEXHy/UENmtNB5ApTkITZ6mEQHVqizVcCJaOtHvwFqVMcVqg6Q4RJ1DYiSub0wgmEgdPlogVZlNLmRDo5KbhnKZz0plAa5i0cs7FGkhnkplp/PaE3ZFxnO6M/vsBYeyEm7sH8dzx1JJiFYXZoWuQ0dMmoS+uX+TFVaCz1TwcvcBYGWyCKblEibfl4zYDL6cm7Yz6srwe+g7MIABFEPAD7us00ohRTh5eKSJVZvaERNFEaVC/pms3fjFmxPUL9YTsU/7fx6p+h5HUhSbbNtjkW7TUbPuK9LV+N2apnW+lxZKXx9ix1FKPSvnJt8Yc6nxvD/OJY6W0KJ3OkziJNOWFzne2AFB6JtIz7l3lUtE1TNqjd7hbqLBan2LHCVAPdXh+wEicbboDEQlCuIeLAIo07sQxrs14IXrIqYQKmuYrPGT253j76Q25f0vNGjM3+tzx9s1bb6iGprPGasBPDdZQOnYpMPFJN1uCZE7vpt7Kui4VWKr2CW2Afu79NPlS4ymqAsqUvmd5i1FHlQhghz4GVcI0ohYyRUuqYfjM2gVAHJ2LcmLZIxjloObWtSlWQddp6sbFTHqojIRpBRrMsPrCOBnO8nnD/93DapyZq5OLl6OJxTISB9cXKUWSn8JyxAfdIdMWCFx8kLuYGx6inO9g8EdfyzgVsTs/gGZHiJQcyLRvJ1Ikz54C41eMUntvOhI78rZ541deEIUJBtr8UwwLvmcyQCiQPo7EPj7T6dTjOcvnXMr4nRv1k2LghPc0XhXjW415qo9Oe6r68qBjTJvIHYuqnDSWZFtR6AFZHo/JSZeJ8nywqnpaINLEcIJ61uNyPSe+1mfn9HtsZpzfLRSh7lu4BhMipgekFyadniKJFyb7h0MAkjor/sLlDKx7RmZsl4dm3pa63EEoex8vY9ahm5jnRmfceYF3Sjz0/cxpELOcCrjmyR+GAzp8nY+odEP3OnI/uGLNOnfKB+ijcpdhdk98NmA/CZ0QsGkdpwNn5G5gz3U93HcqpyUY4rs4NuAxZM5b/heECCQf9Dwrd9xiT8glV/o282z0kCHj/+fu7sBgG4c0YnoNQ0T4JvPn9u/juZwGy5T9MiIsRJ9BhTZIv5AIjecSGqyADquHCyI5gw2UqcSaTgqofDiGPzNXQKN1Ve0Ts/uYfvGmak8PD0s7q5i9iXmhxN8HGeuGNIjBqQll0Hcyqvqqy4lbOBGtEF1KZIkqgfSAqxee6+tp8LEvJ+cszjSI6eo0roGFFmB5aBbM2vgNhztE4pcNub+szx2zWoq/+ZADJH7hmH0tcnrID4E9cuTkwNRCkSHdwUhuyOeGHHasx8HBG5kZzowNRDMv8V6LkpILCDFrn9O5ymyZWL4HpWp15A+F7lk6ffgGlXQXFhqJvpbHxSkkgGXFqERGMnUQxE4Ed7DqdBq4095L6jWXIWW3Mt3bSw1bebbuWOf2U4pT9kpJhW9U0FnrxL9PEHa0AmWrXK+LcVqulkCbHM5wWdICmfweIar/gUuSy5jOkmGvwCoUe8lqDE/5PQ1n1mf6Kr4j0CFZjnSlHe8qPgza4jffiZcB99H83jcAfrWGFEFK8OWeIY68eYdHHRIdbJV2fRzoyra95QRbsgXU4f/FEWKLaJDHp6+k7eJnhyNcddcYkXqnV6DsHfnyMogRmvk/UvqRWpA0Eal1FIQIRml4W6rFphLBJRZDDfCCESm6qLIcaKf2DusrtCgG8h6drrCmDIO5vdICLES+gTpE7wXLOBAunHGu89Rj4lA+u2iKLRpMfDENLzrZ1DOtinp951krMiduI4k8M/kUclmsBp+deLqyj880gTOBAnJi+tRg3CvnjSNMcs3diBKDYugZfijEx0Tmtn3uQdLALnHWHWvsn883dWuj/1wrleWlQNsYPy31dy+YZDoCn7PV0O5Tvw1EybbdxkC/lN99uKZbZz7s6259dRyAh8spndCQL+hbCKcLhxiLMKzvSUtWeBEHr5UV7dTH9Nyw/8Zve4BVeavh+Vuf41wz0KqxZNjee+W3tvKWVSs7lXl6YWq1xBEKLpyblE1LcRqQvjKva75/D7mYoj62rjBayqyB2FzAf72mI4CE18AytlaiRJhtbiEJ2b4LFLA4VfxDyjReCRysZhqPNrWvI9bFh3xzl73PM7Ra3fJ71HQNd99vD3n9zgSu55FXiI9XU7emsyonxh0GPeRQSrMky7tAIwILrMFO1od+sKwVVZNNHTYE2KmNEbdCDeobvT9iWoRCNZZ/8FeNlvWV2hlVZaLoYwFl87ZWgSrX4sFGUvd6RcmSDVep9KpWelkIePIurU/vvE+mRCMfeiXlGHW4CqSuiukUmaBoJjWy61L+OdKlKGZUGcqUdzGF+3q4iLle30slfBpS5TOdXfjLyz/xavDqihj9Abq1r3rknlUDz27Fyda/aj033KDt65xqTk5UrlS36gbmevCsuHfO7FVkAVMmN7rer07wO9NGbuQydvt3afunCKs1F+WtwHLL24g8+atjDiKcb3+C+92skKJ8txN4//DjLex8bCM1Xf0BlHqUS0TKGYAwAF5gJMmjMghQgziCCsw/4ed98U5Yd3PxjMS5WT9TNYR/Fwb1kwTAj1XDpnSxom4CXLMvF5uCro4I9ut9GEB3QqQAy+oQC13vFSIWEL1sk7B2yUHv3zUFxSe3EHvhAxXZJnby2fgdrbbH199k3/zlX8cdK7VPAPAUgHXdCph0uFYJauTewjqfuheIQ3VLgi8GCqRIWGkyMTvwXEWWvrYFgCHJD4b5FSCLW76MtLW96U0FV5uw9mXXbhb6jmI627ZocuwIafyH/736w8EhAU8xwyWrQxdrvDZujRoFr58E27gX5VUb/URKiUrSkzmEThbVWfeaTM9yAFO1lKzz5ImfTWnVVTmdn5g6eXbuaV9v+oaP42w3cVyErKsOmtpAWtIGI4uw+7tBTJlpFMwVtMfhKY+pr4nsdLnztZSc1rMLI56ZLGKyJbxLL8gBPnalj+fcYAITF8zXuwuyIoLdG6bYSbUSObNfZtVOJ0MJ+kOsdtSdRNnL3w11NWZ2vJTjpgQLTtD/mDequLSylrQrOLU+B9eln1dliaS/MgdFzQ2/n8EXOYNUDkwWPxiKS+oiZhnDa2FiPlgArJfV40sSTMPl0gIppRq7jlfYXVqU+CQ68Qq87mCEMfZmzB+R/eY3GZTdgT2uAZIC5PZGZ17Hlrr2Rzags3+dUSnkWvc6XfrjMc97Y+FX8URH1pwc/qWcZHudrQZT+VUpxYFckzNviSosPUnRz38bObG67c+4KhLp8Cl8xCVv2PE2b58jWZzS1yUGUEiDYxyJxe4wGgUQRmTFhQfONqws9HgcUNKrpMm9W4H2izhxzwgsCYhRYO0NS1FfYvmcLoiObk8Tup+B5t4ejlE9qp3Arbn+4VYpdlbguXdEKA9TeW6DN0SoXDkYDifz4XFbhUMcPLdOo+/c2sAXJuqfZuUzhDYFzHF3jGQZggGyA2ntZ1B8Vp79aXLfSDQyn98l1zfb1JQseASovIIReVJe1hbn6S87UH0BQmtXcRGaWCGudP9IxFI32qklw8T8IMrs8A7+JvzrGA1pEFDo+EsYWfXOUihceHVhFz6hBaAc3HkfNXrFzk8YPdLu8NTDgkGt/Zk4GNbCYNdruYQaq7pAiDBrri5DaxoGxysv+V+E0rqhFiRVKvNLCZXNQigtURpBquH7yC0EFCk60ckFEtBHbYvEg96FMUyhYjYkvktXZNFYISvF2u1UO8ToNOIuoIlfHw3vNZRCrRw4TMbz0mZTFDbaviIJEBeQ4xCJfR0SwBUQ3uBwSa5CjHKlNEiHkp5A7748htqbWQxgCIjfaRw0ac4vMrmYPncObddNCiYFXMe0Iaf0O3djXVUXAghqMnKhe/KTntJU2KwvKumYJrLu2o4rF8ItH6enlAYCmxfp+PIPAj/Iojr6DOR+4h+Tn3t4GF94YnjWaWG8qbn2EUQ04a0Kjc7K6mOifhhwBu/Qkq24JFTiFOYyYQKjAYPhq5j36cw5+ZHG3vzEUVYfix8FgwOCMDKveNVgCC1Ydqz/S2Hz/pFaH2hlS6o87YVFJ2VEBrvtMclokgWkAk/hpWIBmqeB9WHBjSwOG1tgZTjBZF2lT6UosxPcL+I3w58ASZ1R2oHsXgp0bHJ8FEF2B+NZQL3OTMlfMz7HHA+VuAsnDQWzWUczifrpqkzfaELBGqzody6xZ1TS06GGMAf4RH/XQWK7Uh6gn1Bd6MK7c6q1eHY7+TNb/eSAsKCaACM0CT7vC63TpvfMFkz6xaO5H9iMeZ/rZ9lz+mLgdgwdys35l/TKgXsxPC3WFrpEyFXwMkFADXlqzACDy5hhCbtGEYDaPcE8RoorDyU6q+w/Ipd/NbDgX7jrXwV4CO4Jzq+59Vbj0DwbWcUTBSlFOKVB1TjQ2yqQGxTGOZQQ9BDaXb0UCtcONyZ4IJfCzUSgXwvmVATdIaUubLvDsUkUMAYPG/zJXsvcYjsP7LPH6dT/SUy7xrU08HS6xIrKnFlE+L9b2Ez+D5lYGf89eQAA"),
}
for filename, (expected, blob) in embedded.items():
    raw = gzip.decompress(base64.b64decode(blob))
    actual = hashlib.sha256(raw).hexdigest()
    if actual != expected:
        raise ValueError(f"Embedded source hash mismatch: {filename}")
    (PACKAGE / filename).write_bytes(raw)
sys.path.insert(0, str(RUNTIME_SRC))
print("Embedded builder source verified:", {k: v[0] for k, v in embedded.items()})


In [ ]:
from findisputeeval.curation.cfpb_seed_v051 import SeedV051Config, build_seed_v051

paths = build_seed_v051(SeedV051Config(
    eda_run_dir=EDA_RUN_DIR,
    decision_record_path=DECISION_RECORD,
    output_dir=OUTPUT_DIR,
    random_seed=20260713,
))
print(json.dumps({name: str(path) for name, path in paths.items()}, indent=2))


In [ ]:
manifest_path = OUTPUT_DIR / "seed_v051_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
errors = []
for name, metadata in manifest["outputs"].items():
    relative = Path(metadata["path"])
    if relative.is_absolute():
        errors.append(f"absolute manifest path: {name}")
        continue
    path = OUTPUT_DIR / relative
    if not path.exists() or sha256_file(path) != metadata["sha256"]:
        errors.append(f"output integrity failure: {name}")
if manifest.get("benchmark_eligible") is not False:
    errors.append("candidate was incorrectly marked benchmark eligible")
if errors:
    raise ValueError("\n".join(errors))
print(json.dumps({
    "release": manifest["release"],
    "status": manifest["release_status"],
    "primary_rows": manifest["primary_rows"],
    "enrichment_rows": manifest["enrichment_rows"],
    "stress_rows": manifest["stress_rows"],
    "excluded_rows": manifest["excluded_rows"],
    "manifest_sha256": sha256_file(manifest_path),
    "output_integrity": "PASS",
    "benchmark_release_gate_passed": manifest["benchmark_release_gate"]["passed"],
}, indent=2))


## Completion boundary

This notebook completes the **candidate build** only. Do not run the formal 50–100 dialogue benchmark pilot until a full Seed + stress NER scan and manual review of all positives are recorded and the benchmark release gate is explicitly changed to passed.
